  
  IMPORTING LIBRARIES

In [2]:
import numpy as np
import pandas as pd
import re #regular expression
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
from nltk.corpus import stopwords

PREPROCESSING

In [4]:
print(stopwords.words('english'))

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [5]:
twitter_data = pd.read_csv('data/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1')

In [6]:
twitter_data.shape

(1599999, 6)

In [7]:
twitter_data.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [8]:
column_names = ['target','id','date','flag','user','text']
twitter_data = pd.read_csv('data/training.1600000.processed.noemoticon.csv',names = column_names, encoding = 'ISO-8859-1')

In [9]:
twitter_data.shape

(1600000, 6)

In [10]:
twitter_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [11]:
twitter_data.isnull().sum()

target    0
id        0
date      0
flag      0
user      0
text      0
dtype: int64

In [12]:
twitter_data['target'].value_counts()

0    800000
4    800000
Name: target, dtype: int64

In [13]:
twitter_data.replace({'target':{4:1}}, inplace=True)

In [14]:
twitter_data['target'].value_counts()

0    800000
1    800000
Name: target, dtype: int64

STEMMING

In [33]:
import numpy as np
import pandas as pd
import re  # regular expression
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from joblib import Parallel, delayed, parallel_backend  # for parallel processing

# Initialize Porter Stemmer
port_stem = PorterStemmer()

# Define stemming function
def stemming(content, stop_words):
    # Apply stemming
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stop_words]
    stemmed_content = ' '.join(stemmed_content)

    return stemmed_content

# Function to apply stemming using parallel processing
def apply_stemming_parallel(data, stop_words, n_jobs=-1):
    # Use all available CPU cores
    with parallel_backend('threading'):
        return Parallel(n_jobs=n_jobs)(delayed(stemming)(content, stop_words) for content in data)

# Assuming twitter_data is your DataFrame
# Split twitter_data into chunks for parallel processing
num_cores = -1  # Use all available CPU cores
chunk_size = 1000  # Adjust the chunk size for optimal performance
chunks = [twitter_data['text'][i:i+chunk_size] for i in range(0, len(twitter_data['text']), chunk_size)]

# Get stopwords list
stop_words = set(stopwords.words('english'))

# Apply stemming using parallel processing
stemmed_contents = []
for chunk in chunks:
    stemmed_contents.extend(apply_stemming_parallel(chunk, stop_words, n_jobs=num_cores))

# Assign stemmed content back to DataFrame
twitter_data['stemmed_content'] = stemmed_contents


In [15]:
import numpy as np
import pandas as pd
import re  # regular expression
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from joblib import Parallel, delayed, parallel_backend  # for parallel processing
import multiprocessing

# Initialize Porter Stemmer
port_stem = PorterStemmer()

# Define stemming function
def stemming(content, stop_words):
    # Apply stemming
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stop_words]
    stemmed_content = ' '.join(stemmed_content)

    return stemmed_content

# Function to apply stemming using parallel processing
def apply_stemming_parallel(data, stop_words, n_jobs=-1):
    # Use all available CPU cores
    with parallel_backend('loky', n_jobs=n_jobs):
        return Parallel()(delayed(stemming)(content, stop_words) for content in data)

# Assuming twitter_data is your DataFrame
# Split twitter_data into chunks for parallel processing
num_cores = multiprocessing.cpu_count()  # Use all available CPU cores
chunk_size = 1000  # Adjust the chunk size for optimal performance
chunks = [twitter_data['text'][i:i+chunk_size] for i in range(0, len(twitter_data['text']), chunk_size)]

# Get stopwords list
stop_words = set(stopwords.words('english'))

# Apply stemming using parallel processing
stemmed_contents = Parallel(n_jobs=num_cores)(delayed(apply_stemming_parallel)(chunk, stop_words, n_jobs=1) for chunk in chunks)

# Flatten the results
stemmed_contents = [item for sublist in stemmed_contents for item in sublist]

# Assign stemmed content back to DataFrame
twitter_data['stemmed_content'] = stemmed_contents

In [16]:
twitter_data.head()

,target,id,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


In [17]:
print(twitter_data['stemmed_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: object


In [18]:
print(twitter_data['target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


TFIDF VECTORIZATION

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time
from joblib import parallel_backend

# Split data into features (X) and target (y)
X = twitter_data['stemmed_content'].values
y = twitter_data['target'].values

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=2)

# Further split training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, stratify=y_train, random_state=2)

# Vectorize the text data
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=2**18)  # Adjust ngram_range and max_features as needed
X_train_vectorized = vectorizer.fit_transform(X_train)
X_val_vectorized = vectorizer.transform(X_val)
X_test_vectorized = vectorizer.transform(X_test)

SVM

In [20]:

# Define and configure the SVM classifier
svm_classifier = LinearSVC(C=0.1, dual=False, max_iter=1000)

# Training
start_time = time.time()
with parallel_backend('threading'):
    svm_classifier.fit(X_train_vectorized, y_train)
svm_train_time = time.time() - start_time

# Cross-validation
cv_start_time = time.time()
cv_scores = cross_val_score(svm_classifier, X_train_vectorized, y_train, cv=5, scoring='accuracy', n_jobs=-1)
cv_time = time.time() - cv_start_time

# Predictions on validation set
svm_val_predictions = svm_classifier.predict(X_val_vectorized)

# Evaluation on validation set
svm_accuracy_val = accuracy_score(y_val, svm_val_predictions)
svm_precision_val = precision_score(y_val, svm_val_predictions)
svm_recall_val = recall_score(y_val, svm_val_predictions)
svm_f1_val = f1_score(y_val, svm_val_predictions)
svm_auc_roc_val = roc_auc_score(y_val, svm_val_predictions)

# Predictions on test set
start_time = time.time()
svm_test_predictions = svm_classifier.predict(X_test_vectorized)
test_time = time.time() - start_time

# Evaluation on test set
svm_accuracy_test = accuracy_score(y_test, svm_test_predictions)
svm_precision_test = precision_score(y_test, svm_test_predictions)
svm_recall_test = recall_score(y_test, svm_test_predictions)
svm_f1_test = f1_score(y_test, svm_test_predictions)
svm_auc_roc_test = roc_auc_score(y_test, svm_test_predictions)

print("Validation Metrics:")
print("Accuracy:", svm_accuracy_val)
print("Precision:", svm_precision_val)
print("Recall:", svm_recall_val)
print("F1 Score:", svm_f1_val)
print("Area Under ROC Curve:", svm_auc_roc_val)
print("Cross-Validation Time:", cv_time)
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

print("\nTest Metrics:")
print("Training Time:", svm_train_time)
print("Test Time:", test_time)
print("Accuracy:", svm_accuracy_test)
print("Precision:", svm_precision_test)
print("Recall:", svm_recall_test)
print("F1 Score:", svm_f1_test)
print("Area Under ROC Curve:", svm_auc_roc_test)


Validation Metrics:
Accuracy: 0.7884791666666666
Precision: 0.7765380979642919
Recall: 0.8100694444444444
F1 Score: 0.7929494456491444
Area Under ROC Curve: 0.7884791666666667
Cross-Validation Time: 275.30960297584534
Cross-Validation Scores: [0.7905054  0.78885031 0.79096065 0.78896219 0.78939043]
Mean CV Accuracy: 0.7897337962962963

Test Metrics:
Training Time: 141.72634553909302
Test Time: 0.2804138660430908
Accuracy: 0.792175
Precision: 0.7809982928999062
Recall: 0.8120625
F1 Score: 0.7962275250946795
Area Under ROC Curve: 0.7921750000000001


LOGISTIC REGRESSION


In [27]:
# Define and configure the logistic regression classifier
logistic_classifier = LogisticRegression(max_iter=1000)

# Training
start_time = time.time()
with parallel_backend('threading', n_jobs=1):
    logistic_classifier.fit(X_train_vectorized, y_train)
logistic_train_time = time.time() - start_time

# Cross-validation
cv_start_time = time.time()
cv_scores = cross_val_score(logistic_classifier, X_train_vectorized, y_train, cv=5, scoring='accuracy', n_jobs=1)
cv_time = time.time() - cv_start_time

# Predictions on validation set
logistic_val_predictions = logistic_classifier.predict(X_val_vectorized)

# Evaluation on validation set
logistic_accuracy_val = accuracy_score(y_val, logistic_val_predictions)
logistic_precision_val = precision_score(y_val, logistic_val_predictions)
logistic_recall_val = recall_score(y_val, logistic_val_predictions)
logistic_f1_val = f1_score(y_val, logistic_val_predictions)
logistic_auc_roc_val = roc_auc_score(y_val, logistic_val_predictions)

# Predictions on test set
start_time = time.time()
logistic_test_predictions = logistic_classifier.predict(X_test_vectorized)
test_time = time.time() - start_time

# Evaluation on test set
logistic_accuracy_test = accuracy_score(y_test, logistic_test_predictions)
logistic_precision_test = precision_score(y_test, logistic_test_predictions)
logistic_recall_test = recall_score(y_test, logistic_test_predictions)
logistic_f1_test = f1_score(y_test, logistic_test_predictions)
logistic_auc_roc_test = roc_auc_score(y_test, logistic_test_predictions)

print("Validation Metrics:")
print("Accuracy:", logistic_accuracy_val)
print("Precision:", logistic_precision_val)
print("Recall:", logistic_recall_val)
print("F1 Score:", logistic_f1_val)
print("Area Under ROC Curve:", logistic_auc_roc_val)
print("Cross-Validation Time:", cv_time)
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

print("\nTest Metrics:")
print("Training Time:", logistic_train_time)
print("Test Time:", test_time)
print("Accuracy:", logistic_accuracy_test)
print("Precision:", logistic_precision_test)
print("Recall:", logistic_recall_test)
print("F1 Score:", logistic_f1_test)
print("Area Under ROC Curve:", logistic_auc_roc_test)

Validation Metrics:
Accuracy: 0.7888472222222223
Precision: 0.778310092872628
Recall: 0.8077777777777778
F1 Score: 0.7927701975110069
Area Under ROC Curve: 0.7888472222222223
Cross-Validation Time: 534.4707548618317
Cross-Validation Scores: [0.79077546 0.78898534 0.79135417 0.78930556 0.78969136]
Mean CV Accuracy: 0.7900223765432098

Test Metrics:
Training Time: 170.66851663589478
Test Time: 0.015624046325683594
Accuracy: 0.7925125
Precision: 0.7829693583883528
Recall: 0.809375
F1 Score: 0.7959532385155319
Area Under ROC Curve: 0.7925125


NAIVE BAYES

In [32]:
from sklearn.naive_bayes import MultinomialNB
# Define and configure the Naive Bayes classifier
nb_classifier = MultinomialNB()

# Training
start_time = time.time()
with parallel_backend('threading', n_jobs=1):
    nb_classifier.fit(X_train_vectorized, y_train)
nb_train_time = time.time() - start_time

# Cross-validation
cv_start_time = time.time()
cv_scores = cross_val_score(nb_classifier, X_train_vectorized, y_train, cv=5, scoring='accuracy', n_jobs=1)
cv_time = time.time() - cv_start_time

# Predictions on validation set
nb_val_predictions = nb_classifier.predict(X_val_vectorized)

# Evaluation on validation set
nb_accuracy_val = accuracy_score(y_val, nb_val_predictions)
nb_precision_val = precision_score(y_val, nb_val_predictions)
nb_recall_val = recall_score(y_val, nb_val_predictions)
nb_f1_val = f1_score(y_val, nb_val_predictions)
nb_auc_roc_val = roc_auc_score(y_val, nb_val_predictions)

# Predictions on test set
start_time = time.time()
nb_test_predictions = nb_classifier.predict(X_test_vectorized)
test_time = time.time() - start_time

# Evaluation on test set
nb_accuracy_test = accuracy_score(y_test, nb_test_predictions)
nb_precision_test = precision_score(y_test, nb_test_predictions)
nb_recall_test = recall_score(y_test, nb_test_predictions)
nb_f1_test = f1_score(y_test, nb_test_predictions)
nb_auc_roc_test = roc_auc_score(y_test, nb_test_predictions)

print("Validation Metrics:")
print("Accuracy:", nb_accuracy_val)
print("Precision:", nb_precision_val)
print("Recall:", nb_recall_val)
print("F1 Score:", nb_f1_val)
print("Area Under ROC Curve:", nb_auc_roc_val)
print("Cross-Validation Time:", cv_time)
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

print("\nTest Metrics:")
print("Training Time:", nb_train_time)
print("Test Time:", test_time)
print("Accuracy:", nb_accuracy_test)
print("Precision:", nb_precision_test)
print("Recall:", nb_recall_test)
print("F1 Score:", nb_f1_test)
print("Area Under ROC Curve:", nb_auc_roc_test)

Validation Metrics:
Accuracy: 0.7705625
Precision: 0.7711235751764067
Recall: 0.7695277777777778
F1 Score: 0.7703248500184219
Area Under ROC Curve: 0.7705625
Cross-Validation Time: 3.8002572059631348
Cross-Validation Scores: [0.77362269 0.77228781 0.7744483  0.77330247 0.77241127]
Mean CV Accuracy: 0.7732145061728395

Test Metrics:
Training Time: 0.7966830730438232
Test Time: 0.05027365684509277
Accuracy: 0.7754625
Precision: 0.7757520396416238
Recall: 0.7749375
F1 Score: 0.7753445558918433
Area Under ROC Curve: 0.7754625


ENSEMBLE LEARNING

In [33]:

from sklearn.ensemble import VotingClassifier

# Define the ensemble classifier
ensemble_classifier = VotingClassifier(estimators=[
    ('svm', svm_classifier),
    ('logistic', logistic_classifier),
    ('nb', nb_classifier)
], voting='hard')

# Training
start_time = time.time()
with parallel_backend('threading', n_jobs=4):
    ensemble_classifier.fit(X_train_vectorized, y_train)
ensemble_train_time = time.time() - start_time

# Cross-validation
cv_start_time = time.time()
cv_scores = cross_val_score(ensemble_classifier, X_train_vectorized, y_train, cv=5, scoring='accuracy', n_jobs=4)
cv_time = time.time() - cv_start_time

# Predictions on validation set
ensemble_val_predictions = ensemble_classifier.predict(X_val_vectorized)

# Evaluation on validation set
ensemble_accuracy_val = accuracy_score(y_val, ensemble_val_predictions)
ensemble_precision_val = precision_score(y_val, ensemble_val_predictions)
ensemble_recall_val = recall_score(y_val, ensemble_val_predictions)
ensemble_f1_val = f1_score(y_val, ensemble_val_predictions)
ensemble_auc_roc_val = roc_auc_score(y_val, ensemble_val_predictions)

# Predictions on test set
start_time = time.time()
ensemble_test_predictions = ensemble_classifier.predict(X_test_vectorized)
test_time = time.time() - start_time

# Evaluation on test set
ensemble_accuracy_test = accuracy_score(y_test, ensemble_test_predictions)
ensemble_precision_test = precision_score(y_test, ensemble_test_predictions)
ensemble_recall_test = recall_score(y_test, ensemble_test_predictions)
ensemble_f1_test = f1_score(y_test, ensemble_test_predictions)
ensemble_auc_roc_test = roc_auc_score(y_test, ensemble_test_predictions)

print("Validation Metrics:")
print("Accuracy:", ensemble_accuracy_val)
print("Precision:", ensemble_precision_val)
print("Recall:", ensemble_recall_val)
print("F1 Score:", ensemble_f1_val)
print("Area Under ROC Curve:", ensemble_auc_roc_val)
print("Cross-Validation Time:", cv_time)
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

print("\nTest Metrics:")
print("Training Time:", ensemble_train_time)
print("Test Time:", test_time)
print("Accuracy:", ensemble_accuracy_test)
print("Precision:", ensemble_precision_test)
print("Recall:", ensemble_recall_test)
print("F1 Score:", ensemble_f1_test)
print("Area Under ROC Curve:", ensemble_auc_roc_test)


Validation Metrics:
Accuracy: 0.7886388888888889
Precision: 0.7775151563710173
Recall: 0.8086805555555555
F1 Score: 0.7927916887926691
Area Under ROC Curve: 0.7886388888888889
Cross-Validation Time: 758.8454637527466
Cross-Validation Scores: [0.79048997 0.7890162  0.79104938 0.78931327 0.78957176]
Mean CV Accuracy: 0.7898881172839507

Test Metrics:
Training Time: 212.75207042694092
Test Time: 0.8677315711975098
Accuracy: 0.79234375
Precision: 0.7820999939690007
Recall: 0.8105
F1 Score: 0.796046775728185
Area Under ROC Curve: 0.79234375


RANDOM FOREST

In [53]:
from sklearn.ensemble import RandomForestClassifier

# Define and configure the Random Forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)

# Training
start_time = time.time()
with parallel_backend('threading'):
    rf_classifier.fit(X_train_vectorized, y_train)
rf_train_time = time.time() - start_time

# Cross-validation
cv_start_time = time.time()
cv_scores = cross_val_score(rf_classifier, X_train_vectorized, y_train, cv=5, scoring='accuracy', n_jobs=-1)
cv_time = time.time() - cv_start_time

# Predictions on validation set
rf_val_predictions = rf_classifier.predict(X_val_vectorized)

# Evaluation on validation set
rf_accuracy_val = accuracy_score(y_val, rf_val_predictions)
rf_precision_val = precision_score(y_val, rf_val_predictions)
rf_recall_val = recall_score(y_val, rf_val_predictions)
rf_f1_val = f1_score(y_val, rf_val_predictions)
rf_auc_roc_val = roc_auc_score(y_val, rf_val_predictions)

# Predictions on test set
start_time = time.time()
rf_test_predictions = rf_classifier.predict(X_test_vectorized)
test_time = time.time() - start_time

# Evaluation on test set
rf_accuracy_test = accuracy_score(y_test, rf_test_predictions)
rf_precision_test = precision_score(y_test, rf_test_predictions)
rf_recall_test = recall_score(y_test, rf_test_predictions)
rf_f1_test = f1_score(y_test, rf_test_predictions)
rf_auc_roc_test = roc_auc_score(y_test, rf_test_predictions)

print("Validation Metrics:")
print("Accuracy:", rf_accuracy_val)
print("Precision:", rf_precision_val)
print("Recall:", rf_recall_val)
print("F1 Score:", rf_f1_val)
print("Area Under ROC Curve:", rf_auc_roc_val)
print("Cross-Validation Time:", cv_time)
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

print("\nTest Metrics:")
print("Training Time:", rf_train_time)
print("Test Time:", test_time)
print("Accuracy:", rf_accuracy_test)
print("Precision:", rf_precision_test)
print("Recall:", rf_recall_test)
print("F1 Score:", rf_f1_test)
print("Area Under ROC Curve:", rf_auc_roc_test)

Validation Metrics:
Accuracy: 0.7273333333333334
Precision: 0.7084883069241351
Recall: 0.7725277777777778
F1 Score: 0.7391235017407713
Area Under ROC Curve: 0.7273333333333334
Cross-Validation Time: 342.4680552482605
Cross-Validation Scores: [0.7298071  0.7285841  0.72968364 0.73055556 0.7264159 ]
Mean CV Accuracy: 0.7290092592592593

Test Metrics:
Training Time: 78.83062601089478
Test Time: 2.420095920562744
Accuracy: 0.73063125
Precision: 0.7121649437116935
Recall: 0.77415
F1 Score: 0.7418649640938412
Area Under ROC Curve: 0.7306312500000001


In [43]:
import joblib

In [54]:
# Save models to disk
joblib.dump(svm_classifier, 'twitter_model_svm.pkl')
joblib.dump(logistic_classifier, 'twitter_model_lr.pkl')
joblib.dump(nb_classifier, 'twitter_model_nb.pkl')
joblib.dump(rf_classifier, 'twitter_model_rfc.pkl')
joblib.dump(ensemble_classifier, 'twitter_model_ensemble.pkl')

['twitter_model_ensemble.pkl']

In [55]:
#Evaluate models on the test set
models = {
    'SVM' : svm_classifier,
    'Logistic Regression' : logistic_classifier,
    'Naive Bayes' : nb_classifier,
    'Random Forest' : rf_classifier,
    'Ensemble' : ensemble_classifier
}

In [56]:
for name, model in models.items():
    start_time = time.time()
    model.fit(X_train_vectorized, y_train)
    training_time = time.time() - start_time
    
    predictions = model.predict(X_test_vectorized)
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, average='weighted')
    recall = recall_score(y_test, predictions, average='weighted')
    f1 = f1_score(y_test, predictions, average='weighted')
    auc = roc_auc_score(y_test, predictions, average='weighted', multi_class='ovr')
    
    print(f"{name}:")
    print(f"  Training Time: {training_time} seconds")
    print(f"  Accuracy: {accuracy*100}%")
    print(f"  Precision: {precision}")
    print(f"  Recall: {recall}")
    print(f"  F1 Score: {f1}")
    print(f"  AUC: {auc}")
    print()


SVM:
  Training Time: 33.41849184036255 seconds
  Accuracy: 79.2175%
  Precision: 0.7926379680803002
  Recall: 0.792175
  F1 Score: 0.7920927700592326
  AUC: 0.7921750000000001

Logistic Regression:
  Training Time: 198.48710083961487 seconds
  Accuracy: 79.25125%
  Precision: 0.7928455754193711
  Recall: 0.7925125
  F1 Score: 0.7924534854133138
  AUC: 0.7925125

Naive Bayes:
  Training Time: 0.8460931777954102 seconds
  Accuracy: 77.54625%
  Precision: 0.7754628036977411
  Recall: 0.7754625
  F1 Score: 0.7754624381118346
  AUC: 0.7754625

Random Forest:
  Training Time: 74.41990232467651 seconds
  Accuracy: 73.063125%
  Precision: 0.7323917397611551
  Recall: 0.73063125
  F1 Score: 0.7301201294785875
  AUC: 0.7306312500000001

Ensemble:
  Training Time: 212.30976223945618 seconds
  Accuracy: 79.234375%
  Precision: 0.7927297427527084
  Recall: 0.79234375
  F1 Score: 0.7922752736656775
  AUC: 0.79234375



In [57]:
# Initialize a dictionary to store performance metrics
performance_metrics = {
    'Model': [],
    'Training Time': [],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1 Score': [],
    'AUC': []
}

# Iterate over each model
models = {
    'SVM' : svm_classifier,
    'Logistic Regression' : logistic_classifier,
    'Naive Bayes' : nb_classifier,
    'Random Forest' : rf_classifier,
    'Ensemble' : ensemble_classifier
}

for name, model in models.items():
    
    start_time = time.time()
    model.fit(X_train_vectorized, y_train)
    training_time = time.time() - start_time
    
    # Make predictions
    predictions = model.predict(X_test_vectorized)
    
    # Compute performance metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, average='weighted')
    recall = recall_score(y_test, predictions, average='weighted')
    f1 = f1_score(y_test, predictions, average='weighted')
    auc = roc_auc_score(y_test, predictions, average='weighted', multi_class='ovr')
    
    print(f"{name}:")
    print(f"  Training Time: {training_time} seconds")
    print(f"  Accuracy: {accuracy*100}%")
    print(f"  Precision: {precision}")
    print(f"  Recall: {recall}")
    print(f"  F1 Score: {f1}")
    print(f"  AUC: {auc}")
    print()
    
    # Add metrics to the dictionary
    performance_metrics['Model'].append(name)
    performance_metrics['Training Time'].append(training_time)
    performance_metrics['Accuracy'].append(accuracy*100)
    performance_metrics['Precision'].append(precision)
    performance_metrics['Recall'].append(recall)
    performance_metrics['F1 Score'].append(f1)
    performance_metrics['AUC'].append(auc)

# Convert dictionary to DataFrame
metrics_df = pd.DataFrame(performance_metrics)

# Save DataFrame to CSV file
metrics_df.to_csv('twitter_performance_metrics.csv', index=False)

SVM:
  Training Time: 41.425292015075684 seconds
  Accuracy: 79.2175%
  Precision: 0.7926379680803002
  Recall: 0.792175
  F1 Score: 0.7920927700592326
  AUC: 0.7921750000000001

Logistic Regression:
  Training Time: 199.03581070899963 seconds
  Accuracy: 79.25125%
  Precision: 0.7928455754193711
  Recall: 0.7925125
  F1 Score: 0.7924534854133138
  AUC: 0.7925125

Naive Bayes:
  Training Time: 0.6815974712371826 seconds
  Accuracy: 77.54625%
  Precision: 0.7754628036977411
  Recall: 0.7754625
  F1 Score: 0.7754624381118346
  AUC: 0.7754625

Random Forest:
  Training Time: 71.30424404144287 seconds
  Accuracy: 73.063125%
  Precision: 0.7323917397611551
  Recall: 0.73063125
  F1 Score: 0.7301201294785875
  AUC: 0.7306312500000001

Ensemble:
  Training Time: 207.33254981040955 seconds
  Accuracy: 79.234375%
  Precision: 0.7927297427527084
  Recall: 0.79234375
  F1 Score: 0.7922752736656775
  AUC: 0.79234375



In [ ]:
import multiprocessing as mp

In [ ]:
print(mp.cpu_count())

8


In [58]:
from sklearn.metrics import confusion_matrix

In [60]:
# Initialize a dictionary to store confusion matrices
confusion_matrices = {}

for name, model in models.items():
    # Make predictions
    predictions = model.predict(X_test_vectorized)
    
    # Compute confusion matrix
    cm = confusion_matrix(y_test, predictions)
    
    # Store confusion matrix in the dictionary
    confusion_matrices[name] = cm
    
    # Save confusion matrix to a pickle file
    filename = f'twitter_{name}_cm.pkl'
    joblib.dump(cm, filename)

In [61]:
from sklearn.metrics import roc_curve

In [62]:
# Initialize a dictionary to store ROC curve data
roc_curve_data = {}

# Iterate over each model
for name, model in models.items():
    # Make predictions
    predictions = model.predict(X_test_vectorized)
    
    # Compute false positive rate, true positive rate, and thresholds
    fpr, tpr, thresholds = roc_curve(y_test, predictions)
    
    # Store ROC curve data in the dictionary
    roc_curve_data[name] = {'fpr': fpr, 'tpr': tpr}

# Save ROC curve data to a Joblib file
joblib.dump(roc_curve_data, 'twitter_roc_curve_data.joblib')

['twitter_roc_curve_data.joblib']